# Считаю retrieval метрики по DINOv2 

In [1]:
import polars as pl
import numpy as np
from pathlib import Path
import chromadb

In [2]:
PROJECT_ROOT_PATH = Path('/Users/a.r.makarenko/Documents/hse/sneakers-hse')

In [3]:
df = pl.read_parquet(PROJECT_ROOT_PATH / "data/04_dinov2_embeddings.parquet.gzip")

embeddings = np.stack(df["embedding"].to_list()).astype("float32")
labels = np.array(df["class"].to_list())
paths = df["path"].to_list()
df['class'].value_counts()

class,count
str,u32
"""PUMA Кроссовки Darter Pro""",135
"""X-Plode Кеды""",270
"""Vans Кеды Upland""",299


In [4]:
client = chromadb.Client()
collection = client.get_or_create_collection(
    "embeddings",
    metadata={"hnsw:space": "cosine"})
collection.add(
    ids=paths,
    embeddings=embeddings,
    metadatas=[{"class": c} for c in labels]
)

In [11]:
def get_neighbors(collection, embeddings, k=10):
    results = collection.query(
        query_embeddings=embeddings,
        n_results=k + 1  # +1 чтобы убрать self
    )['metadatas']
    return np.array([[neighbor['class'] for neighbor in query]
                     for query in results])[:, 1:] #  Убираю self-match

def hit_rate_at_k(neighbors, labels, k):
    hits = 0
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        if query_label in retrieved:
            hits += 1
    return hits / len(labels)

def precision_at_k(neighbors, labels, k):
    precisions = []
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        relevant = (retrieved == query_label).sum()
        precisions.append(relevant / k)
    return np.mean(precisions)

def average_precision(retrieved, query_label, k):
    score = 0.0
    hits = 0
    for i in range(k):
        if retrieved[i] == query_label:
            hits += 1
            score += hits / (i + 1)
    return score / hits if hits > 0 else 0


def map_at_k(neighbors, labels, k):
    scores = []
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        scores.append(average_precision(retrieved, query_label, k))
    return np.mean(scores)
neighbors = get_neighbors(collection, embeddings, k=20)
len(neighbors[0])
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    print("HitRate:", hit_rate_at_k(neighbors, labels, k))
    print("Precision:", precision_at_k(neighbors, labels, k))
    print("MAP:", map_at_k(neighbors, labels, k))


Metrics @ 1
HitRate: 0.9673295454545454
Precision: 0.9673295454545454
MAP: 0.9673295454545454

Metrics @ 5
HitRate: 0.9900568181818182
Precision: 0.922443181818182
MAP: 0.9671855271464644

Metrics @ 10
HitRate: 0.9985795454545454
Precision: 0.8867897727272728
MAP: 0.9507029380039053


In [21]:
df = df.with_columns(neighbors=neighbors)
df = df.with_columns(
    pl.struct(["neighbors", "class"]).map_elements(
        lambda row: [
            1 if n == row["class"] else 0
            for n in row["neighbors"][:k]
        ]
    ).alias("hit_flg")
)
df

path,class,embedding,neighbors,hit_flg
str,str,list[f64],"array[str, 20]",list[i64]
"""Vans Кеды Upland/clothing_0_26…","""Vans Кеды Upland""","[-1.931237, -0.160941, … 0.299092]","[""Vans Кеды Upland"", ""Vans Кеды Upland"", … ""PUMA Кроссовки Darter Pro""]","[1, 1, … 0]"
"""Vans Кеды Upland/clothing_0_57…","""Vans Кеды Upland""","[-1.749463, -1.481841, … -0.965914]","[""Vans Кеды Upland"", ""Vans Кеды Upland"", … ""Vans Кеды Upland""]","[1, 1, … 1]"
"""Vans Кеды Upland/orig_45.jpeg""","""Vans Кеды Upland""","[0.506182, -0.950668, … -3.258668]","[""Vans Кеды Upland"", ""Vans Кеды Upland"", … ""Vans Кеды Upland""]","[1, 1, … 1]"
"""Vans Кеды Upland/clothing_0_0.…","""Vans Кеды Upland""","[-1.44217, -2.105458, … -0.809631]","[""Vans Кеды Upland"", ""Vans Кеды Upland"", … ""X-Plode Кеды""]","[1, 1, … 1]"
"""Vans Кеды Upland/clothing_0_23…","""Vans Кеды Upland""","[-0.842371, -2.197389, … -1.247723]","[""Vans Кеды Upland"", ""Vans Кеды Upland"", … ""Vans Кеды Upland""]","[1, 1, … 1]"
…,…,…,…,…
"""PUMA Кроссовки Darter Pro/orig…","""PUMA Кроссовки Darter Pro""","[-0.281253, -1.683811, … -1.176695]","[""PUMA Кроссовки Darter Pro"", ""PUMA Кроссовки Darter Pro"", … ""PUMA Кроссовки Darter Pro""]","[1, 1, … 1]"
"""PUMA Кроссовки Darter Pro/clot…","""PUMA Кроссовки Darter Pro""","[-1.701043, -1.020201, … -2.74162]","[""PUMA Кроссовки Darter Pro"", ""PUMA Кроссовки Darter Pro"", … ""PUMA Кроссовки Darter Pro""]","[1, 1, … 1]"
"""PUMA Кроссовки Darter Pro/orig…","""PUMA Кроссовки Darter Pro""","[-0.375637, -1.85853, … -1.395184]","[""PUMA Кроссовки Darter Pro"", ""PUMA Кроссовки Darter Pro"", … ""Vans Кеды Upland""]","[1, 1, … 1]"


In [36]:
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    df_k = df.with_columns(hits_k = pl.col("hit_flg").list.slice(0, k))\
        .with_columns(
            pos = pl.int_ranges(1, pl.col("hits_k").list.len() + 1),
            hit_at_k = pl.col("hits_k").list.max(),
            precision_at_k = pl.col("hits_k").list.mean()
        )
    display(df_k.select(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))
    display(df_k.group_by('class').agg(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))


Metrics @ 1


hit_at_k,precision_at_k
f64,f64
0.96733,0.96733


class,hit_at_k,precision_at_k
str,f64,f64
"""X-Plode Кеды""",0.951852,0.951852
"""Vans Кеды Upland""",0.979933,0.979933
"""PUMA Кроссовки Darter Pro""",0.97037,0.97037



Metrics @ 5


hit_at_k,precision_at_k
f64,f64
0.990057,0.922443


class,hit_at_k,precision_at_k
str,f64,f64
"""X-Plode Кеды""",0.977778,0.898519
"""Vans Кеды Upland""",0.996656,0.941137
"""PUMA Кроссовки Darter Pro""",1.0,0.928889



Metrics @ 10


hit_at_k,precision_at_k
f64,f64
0.99858,0.88679


class,hit_at_k,precision_at_k
str,f64,f64
"""X-Plode Кеды""",0.996296,0.854074
"""Vans Кеды Upland""",1.0,0.913712
"""PUMA Кроссовки Darter Pro""",1.0,0.892593
